# Global CLIP Evaluation on MVTec AD
Automated zero-shot evaluation across all 15 categories.

In [ ]:
!pip install "anomalib[vlm,clip]" pandas open_clip_torch

In [ ]:
import pandas as pd
from anomalib.models import WinClip
from anomalib.engine import Engine
from anomalib.data import MVTecAD
import torch

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
CATEGORIES = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid',
    'hazelnut', 'leather', 'metal_nut', 'pill', 'screw',
    'tile', 'toothbrush', 'transistor', 'wood', 'zipper'
]

DATA_ROOT = "/data/mvtec-anomaly-detection"
EVAL_BATCH_SIZE = 128
NUM_WORKERS = 8

all_results = []
engine = Engine()

for category in CATEGORIES:
    print(f"\n{'='*50}")
    print(f"Evaluating category: {category.upper()}")
    print(f"{'='*50}")
    
    datamodule = MVTecAD(
        root=DATA_ROOT,
        category=category,
        eval_batch_size=EVAL_BATCH_SIZE,
        num_workers=NUM_WORKERS
    )
    
    model = WinClip(class_name=category)
    results = engine.test(model=model, datamodule=datamodule)
    
    if results:
        res = results[0] if isinstance(results, list) else results
        res['category'] = category
        all_results.append(res)

if all_results:
    df = pd.DataFrame(all_results)
    display(df)